# 04 — ADME Noise Injection Study (Phase 1: Coarse Scan)

**Goal**: Quantify how RF / LightGBM / FCNN performance degrades under three types of
label noise (Gaussian, systematic bias, gross errors — Landrum & Riniker taxonomy), on the
ADME `hybrid` featureset (FCFP4 + normalized RDKit2D descriptors), for the HLM and SOL
endpoints. See `DECISIONS.md` ADR-005 for the exact noise formulations.

**Featureset note**: this notebook originally ran on `fcfp4` only. Those results are
preserved as `noise_scan_results_fcfp4.csv` / `noise_scan_predictions_fcfp4.pkl` /
`figures/noise_scan_coarse_fcfp4.png`, under the OLD `sigma_frac`/`bias_frac` parameterization
(superseded, see next note) — not regenerated. Checkpoint and figure filenames are suffixed
by `ANALYSIS_FEATURESET` (set in §2) so different featuresets never collide.

**Noise units (amended 2026-08-13)**: Gaussian and systematic bias are now parameterized in
**fold-error** terms (`sigma`/`bias` = `log10(fold)`, absolute log-scale noise), not the old
`sigma_frac`/`bias_frac` (fraction of `std(y)`) — fold-error is scale-invariant across
endpoints for the right reason (it's the *same physical assay noise*, not a fraction of
however spread out an endpoint's labels happen to be) and matches how assay reproducibility
is reported in the ADME literature. Levels: `fold ∈ {2, 3, 5, 10}`, matching
`src/eda/eda.py::FOLD_LEVELS` used by `max_corr_report` (Brown, Muchmore & Hajduk noise
model, via Pat Walters' `useful_rdkit_utils`) — §3.5 below overlays that function's noise
ceiling directly on the Gaussian/bias panels. Gross errors stays in `error_frac` (fraction of
labels corrupted) — fold-error doesn't apply to a categorical corruption process, see §3.5
markdown for why the ceiling overlay is Gaussian/bias-only.

**Design** (mirrors `03_adme_data_quantity.ipynb`'s coarse-then-zoom methodology, see
conversation record / ADR-005 for full rationale):
1. **Phase 1 (this notebook)** — coarse scan: fixed training fraction (1.0, full N), the
   full ADR-005 level grid per noise type, few seeds, baseline (default hyperparameter) arm
   only. Purpose is purely to locate each `(endpoint, model, noise_type)` curve's collapse
   point, not to produce a final publishable curve. The extreme level of each grid (10-fold)
   is a sanity-check anchor — it should land near the `max_corr_report` ceiling, not a
   claimed-realistic operating point.
2. **Phase 2** (later) — identify the collapse point per `(endpoint, model, noise_type)`:
   the same fixed-threshold rule used in `03`'s §5, applied along the noise axis instead of
   the fraction axis.
3. **Phase 3** (later) — zoom in below the identified knee with a finer noise-level grid and
   more seeds, producing the reported degradation curves.

**Metrics**: R² (primary) and MAE (secondary), same choice as `03` — see that notebook's
title cell for the rationale. Pearson r / Spearman / CCC are also recorded (via
`evaluate_model`) but not plotted here.

**Fraction fixed at 1.0**: unlike `03`, this notebook does not vary training-set size — all
fits use the full training pool so that noise is the only manipulated variable. The results
schema still carries a `fraction` column (fixed to `1.0`) purely so the checkpoint schema
stays identical to `03`'s, per the schema-reuse note in that notebook's §2.

## 0 — Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.preprocessing import RobustScaler

from src.models import get_paper_models, evaluate_model, run_checkpointed_eval
from src.noise import add_gaussian_noise, add_systematic_bias, add_gross_errors
from src.plotting import ENDPOINT_LABELS

SEED = 42
DATA_PROC = '../data/processed'
FIGURES = '../figures'
os.makedirs(FIGURES, exist_ok=True)

print('Imports OK')

## 1 — Load existing hybrid splits

Reuses the same `(endpoint, featureset)` splits as `03_adme_data_quantity.ipynb` §1 — no
refeaturization. Each entry carries `X_train`/`X_test` (raw hybrid features — FCFP4 bits +
normalized RDKit2D descriptors concatenated — for RF/LightGBM) and `y_train`/`y_test`. FCNN
also uses raw `X_train`/`X_test` here — its `RobustScaler` is fit fresh on the full training
pool in §3 below (fraction is fixed at 1.0 in this notebook, so there is no per-subsample
refit needed).

In [ ]:
ANALYSIS_FEATURESET = 'hybrid'
EPS = ['HLM', 'SOL']

splits = joblib.load(f'{DATA_PROC}/section4_splits.pkl')
ep_data = {ep: splits[(ep, ANALYSIS_FEATURESET)] for ep in EPS}

for ep in EPS:
    d = ep_data[ep]
    print(f"  {ep}: X_train={d['X_train'].shape}, X_test={d['X_test'].shape}")

## 2 — Config

In [ ]:
FRACTION = 1.0  # fixed -- noise is the only manipulated variable in this notebook
N_SEEDS_COARSE = 15
MODELS = ['RF', 'LightGBM', 'FCNN']
SCALED_MODELS = {'FCNN'}  # models that need RobustScaler'd X

ARM = 'base'  # untuned -- Phase 1 is baseline-only, no tuning arm here

# Coarse-scan levels. Gaussian/systematic_bias are fold-error (see title cell + ADR-005):
# noise_level stores the fold magnitude directly, converted to an absolute log10(fold) sigma/
# bias per endpoint in §3's compute_one (NOT a fraction of std(y) -- that was the old,
# non-scale-invariant sigma_frac/bias_frac parameterization). gross_errors stays in error_frac
# (fraction of labels corrupted) -- fold-error doesn't apply to a categorical corruption
# process. 0-fold/no-noise is covered once by the shared 'none' baseline below rather than
# once per noise type.
NOISE_GRID = {
    'gaussian':         [2, 3, 5, 10],
    'systematic_bias':  [2, 3, 5, 10],
    'gross_errors':     [0.01, 0.05, 0.10, 0.20],
}

n_baseline = len(EPS) * len(MODELS) * N_SEEDS_COARSE
n_noise = len(EPS) * len(MODELS) * sum(len(v) for v in NOISE_GRID.values()) * N_SEEDS_COARSE
print(f'{n_baseline} baseline fits + {n_noise} noise fits = {n_baseline + n_noise} total fits')

## 3 — Coarse sweep

Checkpointed via `run_checkpointed_eval` — re-running this cell only computes missing
`(endpoint, model, fraction, seed, noise_type, noise_level, arm)` keys, so it's safe to
interrupt or extend `NOISE_GRID`/`N_SEEDS_COARSE` later without recomputing everything. The
`KEY_COLS` order matches `03_adme_data_quantity.ipynb` exactly — this is the schema reuse
anticipated in that notebook's §2 config cell.

In [ ]:
RESULTS_CSV = f'{DATA_PROC}/noise_scan_results_{ANALYSIS_FEATURESET}.csv'
PREDICTIONS_PKL = f'{DATA_PROC}/noise_scan_predictions_{ANALYSIS_FEATURESET}.pkl'
KEY_COLS = ('endpoint', 'model', 'fraction', 'seed', 'noise_type', 'noise_level', 'arm')

paper_models = get_paper_models()

def compute_one(key):
    ep, model_name, frac, seed, noise_type, noise_level, arm = key
    d = ep_data[ep]
    scaled = model_name in SCALED_MODELS
    X_train, X_test = d['X_train'], d['X_test']
    y_train, y_test = d['y_train'], d['y_test']

    if noise_type == 'none':
        y_noisy = y_train
    elif noise_type == 'gaussian':
        # noise_level is a fold magnitude -- log10(fold) is the noise sigma in absolute log
        # units (Brown/Muchmore/Hajduk); add_gaussian_noise expects sigma as a FRACTION of
        # std(y), so convert per endpoint here rather than changing that function's contract.
        sigma_frac = np.log10(noise_level) / np.std(y_train)
        y_noisy = add_gaussian_noise(y_train, sigma_frac=sigma_frac, random_state=seed)
    elif noise_type == 'systematic_bias':
        bias_frac = np.log10(noise_level) / np.std(y_train)
        y_noisy = add_systematic_bias(y_train, bias_frac=bias_frac, random_state=seed)
    elif noise_type == 'gross_errors':
        y_noisy = add_gross_errors(y_train, error_frac=noise_level, random_state=seed)
    else:
        raise ValueError(f'Unknown noise_type: {noise_type}')

    X_sub = X_train
    if scaled:
        scaler = RobustScaler().fit(X_sub)
        X_sub = scaler.transform(X_sub)
        X_test = scaler.transform(X_test)

    model = clone(paper_models[model_name])
    model.fit(X_sub, y_noisy)
    y_pred = model.predict(X_test)
    metrics = evaluate_model(None, None, y_test, y_pred=y_pred)

    row = {
        'endpoint': ep, 'model': model_name, 'fraction': frac, 'n_train': len(X_sub),
        'seed': seed, 'noise_type': noise_type, 'noise_level': noise_level, 'arm': arm,
        **metrics,
    }
    pred = {'y_test': y_test, 'y_pred_test': y_pred}
    return row, pred

baseline_keys = [
    (ep, model_name, FRACTION, seed, 'none', 0.0, ARM)
    for ep in EPS
    for model_name in MODELS
    for seed in range(N_SEEDS_COARSE)
]
noise_keys = [
    (ep, model_name, FRACTION, seed, noise_type, level, ARM)
    for ep in EPS
    for model_name in MODELS
    for noise_type, levels in NOISE_GRID.items()
    for level in levels
    for seed in range(N_SEEDS_COARSE)
]
keys = baseline_keys + noise_keys

scan_results, scan_predictions = run_checkpointed_eval(
    keys, compute_one, RESULTS_CSV, PREDICTIONS_PKL, key_cols=KEY_COLS,
)
print(scan_results.shape)
scan_results.head()

## 3.5 — Noise ceiling (Gaussian / systematic bias only)

`max_corr_report` (`src/eda/eda.py`, Brown/Muchmore/Hajduk model via Pat Walters'
`useful_rdkit_utils.max_possible_correlation`) simulates: take the endpoint's real label
values, add one independent Gaussian draw of `sigma = log10(fold)` to each, and compute the
correlation between that single noisy realization and the original. Squared, this gives an
R² ceiling at each fold level.

**What this means on our injected-noise x-axis (corrected 2026-08-17)**: our
`add_gaussian_noise`/`add_systematic_bias` calls perform exactly this operation on
`y_train` at each `noise_level` (fold). So `max_corr_report(fold)` is the correlation ceiling
for a "model" that is a pure lookup table — one that just repeats back the single noisy
training label for a query, with zero denoising or generalization across molecules. Call
this the **memorization ceiling**: it's the best achievable R² *if the model does no
aggregation at all*. A real model (RF/LightGBM/FCNN) trained on thousands of noisy labels can
partially denoise by averaging signal across similar molecules, so it is expected — and
informative, not alarming — for a model's curve to sit *above* this line at high noise
levels: the gap above the memorization ceiling is a direct measure of how much the model
gains from training-set size/aggregation under label noise, which is squarely the point of
this study.

This is *not* Pat Walters' original diagnostic (which asks whether a model's real-world
reported correlation exceeds what `y_test`'s own, real, fixed assay noise permits — a flat
question, unrelated to our synthetic injected-noise axis). We're not asking that question
here; this ceiling is scoped specifically to the synthetic training-noise experiment.

**Not applicable to gross errors** — the ceiling formula assumes i.i.d. Gaussian assay noise;
gross errors are a categorically different corruption process (uniform-random label
replacement on a subset), so the same ceiling number wouldn't describe "best achievable R²
under this kind of noise" for that panel. No overlay there.

Computed on each endpoint's full label pool (`y_train` + `y_test` concatenated) — this is a
property of the endpoint's data, not of the train/test split used for modelling. Not
checkpointed (cheap: seconds, not model fits).

In [ ]:
from src.eda import max_corr_report

FOLD_LEVELS = NOISE_GRID['gaussian']  # reuse the same fold grid as the noise sweep --
# this is the memorization ceiling, scoped to our injected-noise axis, see §3.5 markdown

ceiling_rows = []
for ep in EPS:
    y_full = np.concatenate([ep_data[ep]['y_train'], ep_data[ep]['y_test']])
    report = max_corr_report(pd.DataFrame({ep: y_full}), [ep], fold_levels=FOLD_LEVELS)
    ceiling_rows.append(report)

ceiling_df = pd.concat(ceiling_rows)
ceiling_df

## 4 — Plot: R² and MAE vs noise level

In [ ]:
from scipy import stats

MODEL_COLORS = {'RF': '#1f77b4', 'LightGBM': '#d62728', 'FCNN': '#ff9900'}
NOISE_TYPES = ['gaussian', 'systematic_bias', 'gross_errors']
CEILING_TYPES = {'gaussian', 'systematic_bias'}  # fold-error units only -- see §3.5 markdown

fig, axes = plt.subplots(len(NOISE_TYPES), len(EPS), figsize=(6 * len(EPS), 4 * len(NOISE_TYPES)),
                          sharex=False)

for row_i, noise_type in enumerate(NOISE_TYPES):
    baseline_df = scan_results[scan_results['noise_type'] == 'none'].copy()
    baseline_df['noise_level'] = 0.0
    type_df = scan_results[scan_results['noise_type'] == noise_type]
    plot_df = pd.concat([baseline_df, type_df], ignore_index=True)

    for col, ep in enumerate(EPS):
        ep_df = plot_df[plot_df['endpoint'] == ep]
        ax = axes[row_i, col]

        for model_name in MODELS:
            m_df = ep_df[ep_df['model'] == model_name].groupby('noise_level').agg(
                R2_mean=('R2', 'mean'), R2_std=('R2', 'std'), n_seeds=('R2', 'size'),
            ).reset_index().sort_values('noise_level')
            # 95% CI via t-distribution, not the 1.96 normal shortcut -- n_seeds is only 5
            # per point (df=4), so the t-multiplier (~2.78) is much wider than 1.96.
            t_mult = stats.t.ppf(0.975, df=m_df['n_seeds'] - 1)
            m_df['R2_ci95'] = t_mult * m_df['R2_std'] / np.sqrt(m_df['n_seeds'])

            x = m_df['noise_level']
            color = MODEL_COLORS[model_name]

            ax.plot(x, m_df['R2_mean'], marker='o', label=model_name, color=color)
            ax.fill_between(x, m_df['R2_mean'] - m_df['R2_ci95'], m_df['R2_mean'] + m_df['R2_ci95'],
                            alpha=0.25, color=color)

        if noise_type in CEILING_TYPES:
            # Memorization ceiling: best-achievable R2 for a "model" with zero denoising
            # (a lookup table repeating the single noisy training label) at each injected
            # fold level -- see §3.5. A real model's curve sitting above this line shows it
            # is denoising via aggregation across training molecules.
            ceiling_y = [ceiling_df.loc[ep, f'{fold}-Fold'] for fold in FOLD_LEVELS]
            ax.plot(FOLD_LEVELS, ceiling_y, color='black', linestyle=':', linewidth=1.5,
                    label='memorization ceiling (max_corr_report)')

        ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
        ax.set_title(f'{ENDPOINT_LABELS.get(ep, ep)} -- {noise_type}')
        ax.set_ylabel('R² (test)')
        ax.set_xlabel('fold error' if noise_type in CEILING_TYPES else 'noise level (error_frac)')
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)

fig.suptitle(f'Noise injection coarse scan -- RF / LightGBM / FCNN on {ANALYSIS_FEATURESET}\n'
             f'shaded: 95% CI, n={N_SEEDS_COARSE} seeds/point', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIGURES}/noise_scan_coarse_{ANALYSIS_FEATURESET}.png', dpi=150, bbox_inches='tight')
plt.show()